In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("postgres_silver_validation")
    .master("spark://spark-master:7077")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.driver.memory", "1g")
    .config("spark.sql.autoBroadcastJoinThreshold", "10m")
    .config("spark.sql.parquet.enableVectorizedReader", "true")
    .config("spark.sql.legacy.parquet.nanosAsLong", "true")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minio")
    .config("spark.hadoop.fs.s3a.secret.key", "minio123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/09 21:07:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
POSTGRES_SILVER_PATH = (
    "s3a://end-to-end-streaming-data-platform-silver/"
    "postgres"
)

In [3]:
df_users_silver = spark.read.parquet(
    f"{POSTGRES_SILVER_PATH}/users.parquet"
)

df_subscriptions_silver = spark.read.parquet(
    f"{POSTGRES_SILVER_PATH}/subscription.parquet"
)

df_payments_silver = spark.read.parquet(
    f"{POSTGRES_SILVER_PATH}/payments.parquet"
)

26/09/09 21:08:04 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

In [13]:
df_event_silver = spark.read.parquet(
    "s3a://end-to-end-streaming-data-platform-silver/kafka/events_silver.parquet"
)

df_video_silver = spark.read.parquet(
    "s3a://end-to-end-streaming-data-platform-silver/mongo/videos_silver.parquet"
)

In [24]:
df_event_silver.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- user_id: string (nullable = true)
 |-- video_id: string (nullable = true)
 |-- interaction_type: string (nullable = true)
 |-- watch_time_sec: integer (nullable = true)
 |-- device_type: string (nullable = true)
 |-- event_timestamp_raw: string (nullable = true)
 |-- event_date: date (nullable = true)
 |-- event_hour: integer (nullable = true)
 |-- dq_missing_user_id: boolean (nullable = true)
 |-- dq_missing_video_id: boolean (nullable = true)
 |-- dq_missing_watch_time: boolean (nullable = true)
 |-- dq_unexpected_watch_time: boolean (nullable = true)
 |-- dq_invalid_watch_time: boolean (nullable = true)
 |-- dq_invalid_interaction: boolean (nullable = true)
 |-- dq_invalid_device: boolean (nullable = true)
 |-- dq_invalid_timestamp: boolean (nullable = true)
 |-- ingestion_date: date (nullable = true)
 |-- batch_id: string (nullable = true)



In [15]:
STORAGE_PROTOCOL = "s3a://"
BUCKET_NAME = "end-to-end-streaming-data-platform-gold"
TABLE_NAME_du = "dim_users"
TABLE_NAME_dv = "dim_videos"
TABLE_NAME_fs = "fact_subscription"
TABLE_NAME_fp = "fact_payments"
TABLE_NAME_fe = "fact_events"

In [16]:
gold_dim_users = spark.read.parquet(
    f"{STORAGE_PROTOCOL}{BUCKET_NAME}/{TABLE_NAME_du}"
)
gold_dim_videos = spark.read.parquet(
    f"{STORAGE_PROTOCOL}{BUCKET_NAME}/{TABLE_NAME_dv}"
)
gold_fact_subscriptions = spark.read.parquet(
    f"{STORAGE_PROTOCOL}{BUCKET_NAME}/{TABLE_NAME_fs}"
)
gold_fact_payments = spark.read.parquet(
    f"{STORAGE_PROTOCOL}{BUCKET_NAME}/{TABLE_NAME_fp}"
)
gold_fact_events = spark.read.parquet(
    f"{STORAGE_PROTOCOL}{BUCKET_NAME}/{TABLE_NAME_fe}"
)

In [25]:
gold_fact_events.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- user_id: string (nullable = true)
 |-- video_id: string (nullable = true)
 |-- interaction_type: string (nullable = true)
 |-- watch_time_sec: integer (nullable = true)
 |-- device_type: string (nullable = true)

